# Stitching together `.cha` files from the checkpoints

Perhaps inadvertently, I've saved the outputs for each conversation as a CEDA checkpoint. We need to iterate through them and stitch together the final file for analysis.

In [1]:
import os
import torch
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

In [2]:
DATA_PATH = 'data'

OUTPUT_PATH = os.path.join(DATA_PATH, 'lme_ready')
if not os.path.exists(OUTPUT_PATH):
    os.mkdir(OUTPUT_PATH)

OUTPUT_NAME = os.path.join(OUTPUT_PATH,'grace-ceda-analysis.tsv')

In [3]:
def grab_all_files(PATH, file_type='.cha'):
    files = [
        [
            os.path.join(root, f) for f in files 
            if f.endswith(file_type) and (not f.startswith('._'))
        ] 
        for root, _, files in os.walk(PATH) 
    ]
    return sum(files, [])

## Grabbing each file and creating a merged dataframe

In [4]:
files = [f for f in grab_all_files(DATA_PATH, '.pt') if (not f.startswith('.')) and ('grace_corpus' in f)]
print(files)

['data/graph-obj-grace_corpus.csv.pt']


In [5]:
df_ = []
for f in tqdm(files):
    ckpt = torch.load(f)
    df = pd.DataFrame(ckpt['meta_data'])
    
    # number of tokens per text
    df['nx'] = ckpt['N'][:,0].tolist()
    df['ny'] = ckpt['N'][:,1].tolist()
    
    # CE values
    df['Hxy'] = ckpt['M'][:,0].tolist()
    df['Hyx'] = ckpt['M'][:,1].tolist()
    
    df_ += [df]

df = pd.concat(df_, ignore_index=True)
print(df.shape)
df.head()

  0%|          | 0/1 [00:00<?, ?it/s]

(695015, 14)


,x_speaker,Timestamp,raw_text,conversation_id,corpus,x_turn_id,y_speaker,y_turn_id,y_utterance_delta_no,self,nx,ny,Hxy,Hyx
0,grace-transcript3020-R (M),0:00,Okay. What took a while right?,candor-grace_corpus.csv,grace,0,grace-transcript3020-L (M),1,0,False,8.0,4.0,2.092431,0.664075
1,grace-transcript3020-R (M),0:00,Okay. What took a while right?,candor-grace_corpus.csv,grace,0,grace-transcript3020-L (M),2,1,False,8.0,10.0,2.027158,2.716180
2,grace-transcript3020-R (M),0:00,Okay. What took a while right?,candor-grace_corpus.csv,grace,0,grace-transcript3020-L (M),4,2,False,8.0,9.0,1.991710,2.350171
3,grace-transcript3020-R (M),0:00,Okay. What took a while right?,candor-grace_corpus.csv,grace,0,grace-transcript3020-L (M),6,3,False,8.0,6.0,2.063258,1.354032
4,grace-transcript3020-R (M),0:00,Okay. What took a while right?,candor-grace_corpus.csv,grace,0,grace-transcript3020-L (M),8,4,False,8.0,3.0,2.124930,0.316232


#### Creating Add'l Regression Model Variables

In [ ]:
# df['age_dif'] = df['x_age'] - df['y_age']
# df['pol_dif'] = df['x_politics'] - df['y_politics']
# df['status_dif'] = df['x_i_think_my_status'] - df['x_i_think_your_status']
# df['same_gender'] = (df['x_sex'] == df['y_sex']).astype(int)
# df['same_race'] = (df['x_race'] == df['y_race']).astype(int)
# df['same_edu'] = (df['x_edu'] == df['y_edu']).astype(int)
# df['t_delta'] = df['y_turn_id'] - df['x_turn_id']

In [ ]:
# df[['age_dif', 'pol_dif', 'status_dif', 'same_gender', 'same_race', 't_delta', 'same_edu']].mean()

In [ ]:
# df[['age_dif', 'pol_dif', 'status_dif', 'same_gender', 'same_race', 't_delta', 'same_edu']].median()

In [ ]:
# df[['age_dif', 'pol_dif', 'status_dif', 'same_gender', 'same_race', 't_delta', 'same_edu']].std()

## Save data

In [6]:
df.to_csv(
    OUTPUT_NAME,
    index=False, 
    encoding='utf-8',
    sep='\t'
)